# P6: Operasi Morfologi dan Komponen Terhubung (SIMUTEH)
Bagian 3, 4, dan 5. Jalankan dari folder `notebook/` dengan *Restart & Run All*.

### Kode 1.1 (diulang) Persiapan

In [ ]:
import sys, json
sys.path.append("../src")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from utils_cv import baca, tampil
from praproses import praproses
from utils_tepi import tepi_acuan, skor_tepi, iou, mask_daun_awal, bercak_awal

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
PARAM = json.load(open("../hasil/param_praproses.json"))   # hasil Evaluasi P1-P4
label = pd.read_csv("../data/sintetis/label.csv")

def muat(nama):
    img = baca(f"../data/sintetis/citra/{nama}")
    md = cv2.imread(f"../data/sintetis/mask_daun/{nama}", 0)
    mb = cv2.imread(f"../data/sintetis/mask_bercak/{nama}", 0)
    return img, md, mb

img, md, mb = muat("teh_041.png")
pra = praproses(img, PARAM)
acuan = cv2.bitwise_or(tepi_acuan(md), tepi_acuan(mb))
tampil([img, pra, acuan], ["Citra mentah", "Sesudah praproses", "Tepi acuan (daun + bercak)"])

### Fungsi `tepi_canny` (dari Kode 1.3)

In [ ]:
def tepi_canny(g, lo=30, hi=90, sigma=2.0):
    if sigma > 0:
        g = cv2.GaussianBlur(g, (0, 0), sigma)
    return cv2.Canny(g, lo, hi)

## Bagian 3. Operasi Morfologi pada Citra Biner
**Langkah 3.1 (lembar hitung manual, tulis tangan)**: tidak ada kode.

### Kode 3.1 Mask daun awal dan bercak awal

In [ ]:
def jumlah_komponen(m, konek=8):
    return cv2.connectedComponents((m > 0).astype(np.uint8), connectivity=konek)[0] - 1

baris, panel, judul = [], [], []
for nama in ["teh_041.png", "teh_021.png", "teh_011.png"]:     # berat, ringan, sehat
    img, md, mb = muat(nama)
    pra_ = praproses(img, PARAM)
    m_daun = mask_daun_awal(pra_)
    b0 = bercak_awal(pra_, m_daun, kanal=2, k=4.0)
    baris.append([nama, round(iou(m_daun, md), 3), jumlah_komponen(b0), jumlah_komponen(mb),
                  round(iou(b0, mb), 3) if mb.any() else "-"])
    panel += [pra_, b0, mb]; judul += [nama[:7], "bercak awal", "mask bercak acuan"]
print(pd.DataFrame(baris, columns=["citra", "IoU daun awal", "komponen bercak awal",
                                   "komponen acuan", "IoU bercak awal"]))
tampil(panel, judul, baris=3, ukuran=(14, 10))

### Kode 3.2 Erosi, dilasi, opening, dan closing

In [ ]:
SE = lambda k, bentuk=cv2.MORPH_ELLIPSE: cv2.getStructuringElement(bentuk, (k, k))

def operasi(b, se):
    return {"awal": b,
            "erosi": cv2.erode(b, se),
            "dilasi": cv2.dilate(b, se),
            "opening": cv2.morphologyEx(b, cv2.MORPH_OPEN, se),
            "closing": cv2.morphologyEx(b, cv2.MORPH_CLOSE, se),
            "open+close": cv2.morphologyEx(cv2.morphologyEx(b, cv2.MORPH_OPEN, se), cv2.MORPH_CLOSE, se)}

baris = []
for nama in ["teh_021.png", "teh_011.png"]:
    img, md, mb = muat(nama)
    pra_ = praproses(img, PARAM)
    b0 = bercak_awal(pra_, mask_daun_awal(pra_))
    hasil = operasi(b0, SE(5))
    for nm, m in hasil.items():
        baris.append([nama, nm, jumlah_komponen(m), int((m > 0).sum()),
                      round(iou(m, mb), 3) if mb.any() else "-"])
    tampil(list(hasil.values()), list(hasil.keys()), ukuran=(18, 3)); plt.show()
print(pd.DataFrame(baris, columns=["citra", "operasi", "komponen", "piksel", "IoU"]))

### Kode 3.3 Eksperimen k, bentuk, dan ukuran SE (bahan Tabel 9)

In [ ]:
bentuk = {"persegi": cv2.MORPH_RECT, "elips": cv2.MORPH_ELLIPSE, "silang": cv2.MORPH_CROSS}
baris = []
for _, d in label.iterrows():
    img, md, mb = muat(d.nama_file)
    pra_ = praproses(img, PARAM)
    m_daun = mask_daun_awal(pra_)
    n_acuan = jumlah_komponen(mb)
    for k in [3, 4, 6]:
        b0 = bercak_awal(pra_, m_daun, k=k)
        for nb, kb in bentuk.items():
            for uk in [3, 5, 7]:
                se = SE(uk, kb)
                m = cv2.morphologyEx(cv2.morphologyEx(b0, cv2.MORPH_OPEN, se), cv2.MORPH_CLOSE, se)
                baris.append([d.kelas, k, nb, uk, abs(jumlah_komponen(m) - n_acuan),
                              iou(m, mb) if mb.any() else np.nan])
eks = pd.DataFrame(baris, columns=["kelas", "k", "bentuk", "ukuran", "galat_jumlah", "IoU"])
print(eks.pivot_table(index=["k", "bentuk", "ukuran"], columns="kelas",
                      values=["galat_jumlah", "IoU"]).round(2))
eks.to_csv("../hasil/p6_eksperimen_se.csv", index=False)

### Kode 3.4 Gradien morfologi dibandingkan dengan Canny

In [ ]:
baris = []
for _, d in label.iterrows():
    img, md, mb = muat(d.nama_file)
    pra_ = praproses(img, PARAM)
    m_daun = mask_daun_awal(pra_)
    grad = cv2.morphologyEx(m_daun, cv2.MORPH_GRADIENT, SE(3))     # dilasi - erosi
    canny = tepi_canny(cv2.cvtColor(pra_, cv2.COLOR_BGR2GRAY))
    ad = tepi_acuan(md)
    baris.append([d.kondisi_cahaya, skor_tepi(grad, ad)[2], skor_tepi(canny, ad)[2]])
print(pd.DataFrame(baris, columns=["kondisi", "F1 gradien morfologi", "F1 Canny"])
        .groupby("kondisi").mean().round(3))

### Langkah 3.6 (Kerjakan sendiri) Panel elemen SE

In [ ]:
# Langkah 3.6 (Kerjakan sendiri): elemen SE persegi, elips, silang berukuran 3, 5, 7
nama_bentuk = {"persegi": cv2.MORPH_RECT, "elips": cv2.MORPH_ELLIPSE, "silang": cv2.MORPH_CROSS}
fig, ax = plt.subplots(3, 3, figsize=(7.5, 7.5))
for i, (nb, kb) in enumerate(nama_bentuk.items()):
    for j, uk in enumerate([3, 5, 7]):
        se = cv2.getStructuringElement(kb, (uk, uk))
        ax[i, j].imshow(se, cmap="gray_r", vmin=0, vmax=1)
        ax[i, j].set_title(f"{nb} {uk}x{uk}"); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
        print(f"{nb} {uk}x{uk}:\n{se}\n")
plt.tight_layout()
plt.savefig("../hasil/p6_panel_se.png", dpi=150, bbox_inches="tight")
plt.show()

## Bagian 4. Komponen Terhubung dan Pengukuran Bercak
**Langkah 4.1 (lembar hitung manual, tulis tangan)**: tidak ada kode.

### Kode 4.1 Komponen terhubung dan pengukuran setiap bercak (bahan Tabel 10)

In [ ]:
MM_PER_PX = 0.4375            # dari lembar hitung P2: Z = 200 mm, lebar kerja 640 px

def ukur_komponen(m, konek=8):
    n, lbl, st, cen = cv2.connectedComponentsWithStats((m > 0).astype(np.uint8), connectivity=konek)
    baris = []
    for i in range(1, n):
        kontur, _ = cv2.findContours((lbl == i).astype(np.uint8), cv2.RETR_EXTERNAL,
                                     cv2.CHAIN_APPROX_NONE)
        A = int(st[i, cv2.CC_STAT_AREA])
        keliling = cv2.arcLength(kontur[0], True)
        kebulatan = 0.0 if keliling == 0 else 4 * np.pi * A / keliling ** 2
        w, h = cv2.minAreaRect(kontur[0])[1]
        rasio = max(w, h) / max(min(w, h), 1)
        baris.append([i, A, round(A * MM_PER_PX ** 2, 2), round(keliling, 1),
                      round(kebulatan, 2), round(rasio, 2), int(cen[i][0]), int(cen[i][1])])
    kolom = ["label", "luas_px", "luas_mm2", "keliling_px", "kebulatan", "rasio_aspek", "cx", "cy"]
    return pd.DataFrame(baris, columns=kolom), lbl

img, md, mb = muat("teh_041.png")
pra_ = praproses(img, PARAM)
m_daun = mask_daun_awal(pra_)
bersih = cv2.morphologyEx(cv2.morphologyEx(bercak_awal(pra_, m_daun), cv2.MORPH_OPEN, SE(5)),
                          cv2.MORPH_CLOSE, SE(5))
for konek in [4, 8]:
    print(f"konektivitas {konek}: {jumlah_komponen(bersih, konek)} komponen")
tabel_kom, lbl = ukur_komponen(bersih)
print(tabel_kom)

warna = cv2.applyColorMap((lbl * (255 // max(lbl.max(), 1))).astype(np.uint8), cv2.COLORMAP_JET)
warna[lbl == 0] = pra_[lbl == 0]
for _, r in tabel_kom.iterrows():
    cv2.putText(warna, str(int(r.label)), (int(r.cx) + 8, int(r.cy)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
tampil([pra_, warna], ["teh_041 sesudah praproses", "komponen berlabel"], ukuran=(12, 4.5))
tabel_kom.to_csv("../hasil/p6_ukuran_bercak_teh041.csv", index=False)

### Kode 4.2 Penyaringan komponen berdasarkan ukuran dan bentuk (bahan Tabel 11)

In [ ]:
def saring(m, luas_min=20, bulat_min=0.5, rasio_maks=1.8):
    tabel_k, lbl = ukur_komponen(m)
    lolos = tabel_k[(tabel_k.luas_px >= luas_min) & (tabel_k.kebulatan >= bulat_min)
                    & (tabel_k.rasio_aspek <= rasio_maks)]
    return (np.isin(lbl, lolos.label.values) * 255).astype(np.uint8), lolos

aturan = {"tanpa saring": (0, 0, 99), "luas >= 20": (20, 0, 99),
          "luas + kebulatan": (20, 0.5, 99), "luas + kebulatan + rasio": (20, 0.5, 1.8)}
baris = []
for _, d in label.iterrows():
    img, md, mb = muat(d.nama_file)
    pra_ = praproses(img, PARAM)
    m_daun = mask_daun_awal(pra_)
    bersih = cv2.morphologyEx(cv2.morphologyEx(bercak_awal(pra_, m_daun), cv2.MORPH_OPEN, SE(5)),
                              cv2.MORPH_CLOSE, SE(5))
    for nm, (a, b, r) in aturan.items():
        f, _ = saring(bersih, a, b, r)
        baris.append([d.kelas, nm, jumlah_komponen(f), jumlah_komponen(mb)])
uji = pd.DataFrame(baris, columns=["kelas", "aturan", "jumlah", "acuan"])
uji["galat"] = (uji.jumlah - uji.acuan).abs()
print(uji.pivot_table(index="aturan", columns="kelas", values="galat").round(2))
print("Daun sehat yang masih memiliki bercak palsu:")
print(uji[uji.kelas == "sehat"].assign(palsu=lambda t: t.jumlah > 0).groupby("aturan").palsu.sum())

### Langkah 4.4 (Kerjakan sendiri) Diagram sebar komponen benar dan palsu

In [ ]:
# Langkah 4.4 (Kerjakan sendiri): dasar ambang penyaringan (kebulatan vs rasio aspek)
baris = []
for _, d in label.iterrows():
    img, md, mb = muat(d.nama_file)
    pra_ = praproses(img, PARAM)
    m_daun = mask_daun_awal(pra_)
    bersih = cv2.morphologyEx(cv2.morphologyEx(bercak_awal(pra_, m_daun), cv2.MORPH_OPEN, SE(5)),
                              cv2.MORPH_CLOSE, SE(5))
    tabel_k, lbl = ukur_komponen(bersih)
    for _, r in tabel_k.iterrows():
        di_acuan = (mb[lbl == r.label] > 0).mean()            # porsi piksel komponen di mask acuan
        baris.append([d.nama_file, r.luas_px, r.kebulatan, r.rasio_aspek, di_acuan > 0.5])
kom = pd.DataFrame(baris, columns=["citra", "luas_px", "kebulatan", "rasio_aspek", "benar"])
print(kom.groupby("benar")[["luas_px", "kebulatan", "rasio_aspek"]].describe().round(2).T)

plt.figure(figsize=(6.5, 5))
for nilai, warna_, nama_ in [(True, "tab:green", "komponen benar"), (False, "tab:red", "komponen palsu")]:
    s = kom[kom.benar == nilai]
    plt.scatter(s.kebulatan, s.rasio_aspek, c=warna_, s=22, alpha=0.7, label=f"{nama_} (n={len(s)})")
plt.axvline(0.5, ls="--", c="k"); plt.axhline(1.8, ls=":", c="k")
plt.xlabel("kebulatan"); plt.ylabel("rasio aspek"); plt.legend()
plt.title("Komponen benar dan palsu (60 citra)")
plt.savefig("../hasil/p6_sebar_komponen.png", dpi=150, bbox_inches="tight")
plt.show()

## Bagian 5. Integrasi: Modul `src/bercak.py`
Modul `bercak.py` (tujuh TODO) ada di `src/bercak.py`. Notebook ini hanya memanggil dan mengevaluasinya.

### Kode 5.2 Evaluasi modul bercak.py (bahan Tabel 12)

In [ ]:
import importlib, bercak
importlib.reload(bercak)
PB = {}                                    # ubah parameter sesuai temuan Anda
bercak.simpan_param(PB)

baris = []
for _, d in label.iterrows():
    img, md, mb = muat(d.nama_file)
    h = bercak.analisis_daun(praproses(img, PARAM), PB)
    baris.append([d.nama_file, d.kelas, d.kondisi_cahaya, d.persen_bercak, h["persen_bercak"],
                  jumlah_komponen(mb), h["jumlah_bercak"], h["kelas"]])
ev = pd.DataFrame(baris, columns=["nama", "kelas", "kondisi", "persen_acuan", "persen_prediksi",
                                  "jumlah_acuan", "jumlah_prediksi", "kelas_prediksi"])
ev["kelas_acuan"] = ev.kelas.map({"sehat": "Sehat", "ringan": "Bercak Ringan", "berat": "Bercak Berat"})
ev.to_csv("../hasil/p6_evaluasi_bercak.csv", index=False)

print("Galat absolut rerata persen bercak:", (ev.persen_prediksi - ev.persen_acuan).abs().mean().round(3))
print("Korelasi persen prediksi vs acuan  :", ev[["persen_prediksi", "persen_acuan"]].corr().iloc[0, 1].round(3))
print("Akurasi kelas                      :", (ev.kelas_prediksi == ev.kelas_acuan).mean().round(3))
print(pd.crosstab(ev.kelas_acuan, ev.kelas_prediksi, rownames=["acuan"], colnames=["prediksi"]))
print(ev.assign(benar=ev.kelas_prediksi == ev.kelas_acuan).groupby("kondisi").benar.mean().round(2))

plt.figure(figsize=(5, 5))
plt.scatter(ev.persen_acuan, ev.persen_prediksi, c=ev.kelas.astype("category").cat.codes)
plt.plot([0, 8], [0, 8], "k--"); plt.xlabel("persen bercak acuan"); plt.ylabel("persen bercak prediksi")
plt.savefig("../hasil/p6_scatter_persen.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Bahan Tabel 12: jumlah daun sehat dinilai berbercak, dan daun berbercak dinilai sehat
sehat_palsu = int(((ev.kelas_acuan == "Sehat") & (ev.kelas_prediksi != "Sehat")).sum())
berbercak_lolos = int(((ev.kelas_acuan != "Sehat") & (ev.kelas_prediksi == "Sehat")).sum())
kond_terendah = ev.assign(benar=ev.kelas_prediksi == ev.kelas_acuan).groupby("kondisi").benar.mean().idxmin()
print("Daun sehat dinilai berbercak :", sehat_palsu)
print("Daun berbercak dinilai sehat :", berbercak_lolos)
print("Kondisi cahaya akurasi terendah:", kond_terendah)

### Langkah 5.3 (Kerjakan sendiri) Analisis kesalahan

In [ ]:
# Langkah 5.3 (Kerjakan sendiri): analisis kesalahan. Tampilkan citra yang salah kelas
salah = ev[ev.kelas_prediksi != ev.kelas_acuan]
print(f"{len(salah)} citra salah kelas dari {len(ev)}")
print(salah[["nama", "kondisi", "persen_acuan", "persen_prediksi", "kelas_acuan", "kelas_prediksi",
             "jumlah_acuan", "jumlah_prediksi"]].to_string(index=False))
for _, d in salah.iterrows():
    img, md, mb = muat(d.nama)
    pra_ = praproses(img, PARAM)
    h = bercak.analisis_daun(pra_, PB)
    b0 = bercak_awal(pra_, h["mask_daun"], kanal=2, k=4.0)
    tampil([pra_, b0, h["mask_bercak"], mb],
           [f"{d.nama} ({d.kondisi})", "bercak awal", f"hasil: {d.persen_prediksi:.2f}%", f"acuan: {d.persen_acuan:.2f}%"],
           ukuran=(16, 3.4))
    plt.show()

### Langkah 5.4 Justifikasi parameter (Tabel 13)

In [ ]:
# Langkah 5.4: parameter final yang tersimpan
print(open("../hasil/param_bercak.json").read())

### Langkah 5.5 Reproduksibilitas
Jalankan ulang dengan *Restart & Run All*, perbarui `README.md`, commit, lalu `git tag v0.3-morfologi` dan `git push --tags`.